In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os
warnings.filterwarnings('ignore')

# Spoločný štýl pre všetky grafy v notebooku
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.dpi'] = 100

# === KONFIGURÁCIA - UPRAV PODĽA POTREBY ===
# Cesty k exportovaným dátam z pipeline
FORECAST_10MIN = '../Data/Export_data/forecast_10min.parquet'
FORECAST_1H = '../Data/Export_data/forecast_1h.parquet'
SEGMENTS_FILE = '../Data/Export_data/segments.csv'

# Trojfázová sústava - 3 napätia a 3 prúdy
voltage_cols = ['u1_norm', 'u2_norm', 'u3_norm']
current_cols = ['i1_norm', 'i2_norm', 'i3_norm']

# Sem sa budú ukladať všetky vygenerované grafy
OUTPUT_DIR = '../Data/Export_data/Obrazky'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Konfigurácia načítaná.")
print(f"Výstupný adresár: {OUTPUT_DIR}/")

---
# ČASŤ 1: DOKUMENTÁCIA ZDROJOVÝCH DÁT A PIPELINE
---

Táto sekcia dokumentuje štruktúru pôvodných datasetov a proces ich spracovania.

In [ ]:
print("="*70)
print("ČASŤ 1: POPIS ZDROJOVÝCH DÁT")
print("="*70)

print("""
================================================================================
                         ŠTRUKTÚRA ZDROJOVÝCH DÁT
================================================================================

DATASET: tuke_poruchy_vzorky (Data/Druhe_data/)
────────────────────────────────────────────────────────────────────────────────
Súbor meraní: tuke_poruchy_vzorky.parquet
  • Stĺpce: t_utc, elektromer, u1_norm, u2_norm, u3_norm, 
            i1_norm, i2_norm, i3_norm

Súbor porúch: tuke_poruchy_vzorky.xlsx (CSV formát)
  • Stĺpce: eic, elektromer, contract_id, porucha_eic_od, 
            porucha_eic_do, data_od, data_do, platnost_elm_od, platnost_elm_do

Charakteristika:
  • Dataset obsahuje vzorky elektromerov s evidovanými poruchami
  • Poruchy priradené podľa dátumových rozsahov (porucha_eic_od – porucha_eic_do)
  • Príznak Chyba: 1 ak meranie spadá do poruchového obdobia, 0 inak

================================================================================
""")


In [ ]:
print("="*70)
print("PROCES SPRACOVANIA DÁT (PIPELINE)")
print("="*70)

print("""
================================================================================
                         PIPELINE SPRACOVANIA DÁT
================================================================================

KROK 1: NAČÍTANIE A PRÍPRAVA DÁT
────────────────────────────────────────────────────────────────────────────────
  1.1 Načítanie parquet súboru s meraniami
  1.2 Načítanie CSV súboru s evidenciou porúch
  1.3 Priradenie príznaku Chyba pre každé meranie:
      • Pre každý záznam poruchy z CSV: označenie meraní daného
        elektromera v intervale [porucha_eic_od, porucha_eic_do] ako Chyba=1
      • Všetky ostatné merania: Chyba=0
  1.4 Priradenie EIC kódov z CSV k elektromerom
  1.5 Čistenie dát (odstránenie neplatných EIC)

KROK 2: FILTROVANIE VN ELEKTROMEROV
────────────────────────────────────────────────────────────────────────────────
  2.1 Určenie typu siete podľa mediánu napätia:
      • VN (vysoké napätie): medián > 500V
      • NN (nízke napätie): medián ≤ 500V
  2.2 Odstránenie VN elektromerov (pracujeme len s NN)

KROK 3: OPRAVA HODNÔT
────────────────────────────────────────────────────────────────────────────────
  3.1 Oprava napätia (nulové hodnoty a NaN):
      • Metóda: lineárna interpolácia (limit 24 krokov)
      • Fallback: forward/backward fill
      • Posledná záchrana: medián elektromera / globálny medián
  
  3.2 Oprava prúdu (len NaN, nulové zostávajú):
      • Metóda: lineárna interpolácia + fill
      • Fallback: 0

KROK 4: ODSTRÁNENIE OUTLIEROV
────────────────────────────────────────────────────────────────────────────────
  4.1 IQR metóda s k=3.0
  4.2 Odstránenie celých riadkov obsahujúcich outlier

KROK 5: SEGMENTÁCIA
────────────────────────────────────────────────────────────────────────────────
  5.1 Identifikácia prerušení:
      • Časová medzera > 20 minút (2× frekvencia merania)
      • Zmena stavu príznaku Chyba
  5.2 Priradenie segment_id pre každý súvislý úsek
  5.3 Filtrácia použiteľných segmentov (Chyba=0):
      • Minimálna dĺžka: 8 dní (7d tréning + 1d forecast)

KROK 6: AGREGÁCIA A EXPORT
────────────────────────────────────────────────────────────────────────────────
  Exportované dáta obsahujú AJ normálne AJ poruchové záznamy:
      • Chyba=0: platné segmenty (tréning + validácia modelov)
      • Chyba=1: poruchové záznamy (testovanie anomaly detection)
  
  Metóda agregácie: medián pre napätie/prúd, max pre Chyba
  
  Výstupné granularity: 10 minút, 30 minút, 1 hodina

================================================================================
""")


In [ ]:
# Vizualizácia pipeline ako blokový diagram
fig, ax = plt.subplots(figsize=(16, 14))
ax.set_xlim(0, 14)
ax.set_ylim(-1.5, 16)   # rozšírený rozsah zdola, aby sa výstupné boxy nezarezávali
ax.axis('off')

# Farby pre jednotlivé typy blokov
colors = {'input': '#BBDEFB', 'process': '#FFF9C4', 'output': '#C8E6C9',
          'arrow': '#455A64', 'fault': '#FFCDD2'}

# Pomocná funkcia - vykreslí jeden farebný obdĺžnik s textom v strede
def draw_box(ax, x, y, w, h, text, color, fontsize=14, fontweight='normal'):
    rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor='black',
                         linewidth=1.8, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight=fontweight, wrap=True)

# Titulok
ax.text(7, 15.5, 'PIPELINE SPRACOVANIA DÁT', ha='center',
        fontsize=30, fontweight='bold')

# Vstupné dáta - parquet s meraniami a CSV s poruchami
draw_box(ax, 1.0, 12.8, 5.0, 2.0,
         'PARQUET\nMerania elektromerov\n(10min granularita)',
         colors['input'], fontsize=20)
draw_box(ax, 8.0, 12.8, 5.0, 2.0,
         'CSV\nEvidencia porúch\n(dátumové rozsahy)',
         colors['fault'], fontsize=20)

# Šípky z oboch vstupov sa zbiehajú do prvého procesného bloku
ax.annotate('', xy=(7, 11.3), xytext=(3.5, 12.8),
            arrowprops=dict(arrowstyle='->', color=colors['arrow'], lw=2.5))
ax.annotate('', xy=(7, 11.3), xytext=(10.5, 12.8),
            arrowprops=dict(arrowstyle='->', color=colors['arrow'], lw=2.5))

# Proces 1: Priradenie porúch
draw_box(ax, 3.5, 9.7, 7.0, 1.6,
         'PRIRADENIE PORÚCH\nChyba=0 / Chyba=1',
         colors['process'], fontsize=20)
ax.annotate('', xy=(7, 8.4), xytext=(7, 9.7),
            arrowprops=dict(arrowstyle='->', color=colors['arrow'], lw=2.5))

# Proces 2: VN filter + oprava
draw_box(ax, 3.5, 6.8, 7.0, 1.6,
         'VN FILTER + OPRAVA HODNÔT\nOdstránenie VN, interpolácia, IQR',
         colors['process'], fontsize=20)
ax.annotate('', xy=(7, 5.5), xytext=(7, 6.8),
            arrowprops=dict(arrowstyle='->', color=colors['arrow'], lw=2.5))

# Proces 3: Segmentácia
draw_box(ax, 3.5, 3.9, 7.0, 1.6,
         'SEGMENTÁCIA\nPodľa času + zmeny Chyba',
         colors['process'], fontsize=20)
ax.annotate('', xy=(7, 2.6), xytext=(7, 3.9),
            arrowprops=dict(arrowstyle='->', color=colors['arrow'], lw=2.5))

# Proces 4: Agregácia + Export
draw_box(ax, 3.5, 1.0, 7.0, 1.6,
         'AGREGÁCIA + EXPORT\n(Chyba=0 tréning + Chyba=1 testovanie)',
         colors['process'], fontsize=20)

# Šípky do výstupov - jedna do každej granularity
ax.annotate('', xy=(2.0, -0.1), xytext=(5.0, 1.0),
            arrowprops=dict(arrowstyle='->', color=colors['arrow'], lw=2))
ax.annotate('', xy=(7.0, -0.1), xytext=(7.0, 1.0),
            arrowprops=dict(arrowstyle='->', color=colors['arrow'], lw=2))
ax.annotate('', xy=(12.0, -0.1), xytext=(9.0, 1.0),
            arrowprops=dict(arrowstyle='->', color=colors['arrow'], lw=2))

# Tri výstupné granularity
draw_box(ax, 0.3, -1.3, 3.5, 1.2, '10 min',   colors['output'], fontsize=20, fontweight='bold')
draw_box(ax, 5.25, -1.3, 3.5, 1.2, '30 min',  colors['output'], fontsize=20, fontweight='bold')
draw_box(ax, 10.2, -1.3, 3.5, 1.2, '1 hodina', colors['output'], fontsize=20, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_pipeline_diagram.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()
print(f"Uložené: {OUTPUT_DIR}/01_pipeline_diagram.png")


---
# ČASŤ 2: NAČÍTANIE SPRACOVANÝCH DÁT
---

In [ ]:
print("="*70)
print("ČASŤ 2: NAČÍTANIE SPRACOVANÝCH DÁT")
print("="*70)

# Načítanie hlavného datasetu (1-hodinová granularita)
print(f"\nNačítavam: {FORECAST_1H}")
df = pd.read_parquet(FORECAST_1H)
df['t_utc'] = pd.to_datetime(df['t_utc'])
print(f"Načítané: {len(df):,} riadkov")

# Koľko máme normálnych a koľko poruchových záznamov
n_clean = (df['Chyba'] == 0).sum()
n_fault = (df['Chyba'] == 1).sum()
print(f"\nRozdelenie podľa príznaku Chyba:")
print(f"  Chyba=0 (normálne):   {n_clean:,} ({n_clean/len(df)*100:.1f}%)")
print(f"  Chyba=1 (poruchy):    {n_fault:,} ({n_fault/len(df)*100:.1f}%)")

# Metadáta o segmentoch (CSV používa ';' ako oddeľovač)
print(f"\nNačítavam: {SEGMENTS_FILE}")
segments = pd.read_csv(SEGMENTS_FILE, sep=';')
print(f"Segmentov: {len(segments):,}")

print(f"\nPamäť: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")


In [ ]:
print("\n" + "="*70)
print("ZÁKLADNÝ PREHĽAD")
print("="*70)

print(f"""
SPRACOVANÝ DATASET (1h granularita):
  Počet záznamov: {len(df):,}
  Počet unikátnych EIC: {df['eic'].nunique():,}
  Časový rozsah: {df['t_utc'].min()} až {df['t_utc'].max()}
  Celková dĺžka: {(df['t_utc'].max() - df['t_utc'].min()).days:,} dní

  Chyba=0 (normálne): {(df['Chyba']==0).sum():,} záznamov
  Chyba=1 (poruchy):  {(df['Chyba']==1).sum():,} záznamov
  EIC s poruchami:     {df[df['Chyba']==1]['eic'].nunique()} elektromerov
""")

print("Stĺpce:")
print(f"  {list(df.columns)}")

print("\nDátové typy:")
print(df.dtypes.to_string())

print("\nUkážka dát:")
display(df.head())


---
# ČASŤ 3: ROZDELENIE VN vs NN
---

In [ ]:
print("="*70)
print("ČASŤ 3: OVERENIE TYPU SIETE")
print("="*70)

# Skontrolujeme, či dataset naozaj obsahuje len NN elektromery (VN sa odfiltrovalo v pipeline)
if 'typ_siete' in df.columns:
    type_stats = df['typ_siete'].value_counts()
    print(f"\nTypy siete v dátach:")
    for typ, count in type_stats.items():
        print(f"  {typ}: {count:,} záznamov ({count/len(df)*100:.1f}%)")
else:
    print("\nStĺpec typ_siete nie je prítomný - dáta obsahujú len NN elektromery")
    print(f"Celkovo záznamov: {len(df):,}")

# Rozdelíme dáta na čisté (na trénovanie) a poruchové (na test anomaly detection)
nn_data = df[df['Chyba'] == 0].copy()
vn_data = pd.DataFrame()
fault_data = df[df['Chyba'] == 1].copy()

print(f"\nNN dáta pre analýzu (Chyba=0): {len(nn_data):,} záznamov")
print(f"Poruchové dáta (Chyba=1): {len(fault_data):,} záznamov")


In [ ]:
# Vizualizácia - distribúcia napätia (potvrdenie NN)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sample namiesto celých dát - histogram z miliónov hodnôt by sa kreslil zbytočne dlho
sample_plot = nn_data.sample(min(100000, len(nn_data)), random_state=42)

# Histogram napätia U1 - peak by mal byť okolo 230V (potvrdenie NN siete)
axes[0].hist(sample_plot['u1_norm'], bins=100, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_title('Distribúcia napätia U1 (NN)')
axes[0].set_xlabel('Napätie (V)')
axes[0].set_ylabel('Počet')

# Histogram prúdu I1
axes[1].hist(sample_plot['i1_norm'], bins=100, color='coral', alpha=0.7, edgecolor='black')
axes[1].set_title('Distribúcia prúdu I1 (NN)')
axes[1].set_xlabel('Prúd (A)')
axes[1].set_ylabel('Počet')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/distribúcia_nn.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Uložené: {OUTPUT_DIR}/distribúcia_nn.png")


---
# ČASŤ 4: POPISNÉ ŠTATISTIKY
---

In [ ]:
print("="*70)
print("ČASŤ 4: POPISNÉ ŠTATISTIKY")
print("="*70)

# describe() s extra percentilmi 1% a 99% - tie ukazujú extrémne hodnoty
print("\n--- CELKOVÉ ŠTATISTIKY (všetky dáta) ---")
stats_all = df[voltage_cols + current_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
print(stats_all.round(4).to_string())

In [ ]:
print("\n--- ŠTATISTIKY PRE NN ELEKTROMERY ---")
if len(nn_data) > 0:
    stats_nn = nn_data[voltage_cols + current_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    print(stats_nn.round(4).to_string())

In [ ]:
print("\n--- ŠTATISTIKY PRE VN ELEKTROMERY ---")
if len(vn_data) > 0:
    stats_vn = vn_data[voltage_cols + current_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    print(stats_vn.round(4).to_string())
else:
    print("Nedostatok VN dát.")

---
# ČASŤ 5: ČASOVÁ ANALÝZA
---

In [ ]:
print("="*70)
print("ČASŤ 5: ČASOVÁ ANALÝZA")
print("="*70)

# Z časovej pečiatky vytiahneme hodinu, deň v týždni, mesiac a rok
# - potrebujeme to na agregáciu a hľadanie denných/sezónnych vzorov
df['hour'] = df['t_utc'].dt.hour
df['dayofweek'] = df['t_utc'].dt.dayofweek
df['month'] = df['t_utc'].dt.month
df['year'] = df['t_utc'].dt.year

if len(nn_data) > 0:
    nn_data = nn_data.copy()
    nn_data['hour'] = nn_data['t_utc'].dt.hour
    nn_data['dayofweek'] = nn_data['t_utc'].dt.dayofweek
    nn_data['month'] = nn_data['t_utc'].dt.month

print(f"\nČasový rozsah: {df['t_utc'].min()} až {df['t_utc'].max()}")
print(f"Celková dĺžka: {(df['t_utc'].max() - df['t_utc'].min()).days} dní")

In [ ]:
# Štyri pohľady na to, ako sú dáta rozložené v čase
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Podľa hodiny - mal by byť zhruba rovnomerný (24 stĺpčekov plus-mínus rovnako vysokých)
hourly_counts = df.groupby('hour').size()
axes[0, 0].bar(hourly_counts.index, hourly_counts.values, color='steelblue', edgecolor='black')
axes[0, 0].set_xlabel('Hodina dňa')
axes[0, 0].set_ylabel('Počet záznamov')
axes[0, 0].set_title('Distribúcia záznamov podľa hodiny')
axes[0, 0].set_xticks(range(24))

# Podľa dňa v týždni
daily_counts = df.groupby('dayofweek').size()
days = ['Po', 'Ut', 'St', 'Št', 'Pi', 'So', 'Ne']
axes[0, 1].bar(range(7), daily_counts.values, color='coral', edgecolor='black')
axes[0, 1].set_xlabel('Deň v týždni')
axes[0, 1].set_ylabel('Počet záznamov')
axes[0, 1].set_title('Distribúcia záznamov podľa dňa v týždni')
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(days)

# Podľa mesiaca - tu uvidíme, či dataset nepokrýva niektoré mesiace nedostatočne
monthly_counts = df.groupby('month').size()
axes[1, 0].bar(monthly_counts.index, monthly_counts.values, color='seagreen', edgecolor='black')
axes[1, 0].set_xlabel('Mesiac')
axes[1, 0].set_ylabel('Počet záznamov')
axes[1, 0].set_title('Distribúcia záznamov podľa mesiaca')
axes[1, 0].set_xticks(range(1, 13))

# Podľa roku
yearly_counts = df.groupby('year').size()
axes[1, 1].bar(yearly_counts.index.astype(str), yearly_counts.values, color='purple', edgecolor='black')
axes[1, 1].set_xlabel('Rok')
axes[1, 1].set_ylabel('Počet záznamov')
axes[1, 1].set_title('Distribúcia záznamov podľa roku')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_casova_distribucia.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Uložené: {OUTPUT_DIR}/03_casova_distribucia.png")

In [ ]:
# Denný profil - ako sa priemerné hodnoty napätia a prúdu menia počas dňa
# Pri prúde očakávame špičky ráno a večer
print("\n--- DENNÝ PROFIL (NN elektromery) ---")

if len(nn_data) > 0:
    # Pre každú hodinu spočítame priemer cez všetky elektromery a všetky dni
    hourly_voltage = nn_data.groupby('hour')[voltage_cols].mean()
    hourly_current = nn_data.groupby('hour')[current_cols].mean()
    
    print("\nPriemerné napätie podľa hodiny:")
    print(hourly_voltage.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Napätie
for col in voltage_cols:
    axes[0].plot(hourly_voltage.index, hourly_voltage[col], marker='o', label=col, linewidth=2)
axes[0].set_xlabel('Hodina dňa')
axes[0].set_ylabel('Priemerné napätie [V]')
axes[0].set_title('Denný profil napätia (NN elektromery)')
axes[0].legend()
axes[0].set_xticks(range(24))
axes[0].grid(True, alpha=0.3)

# Prúd
for col in current_cols:
    axes[1].plot(hourly_current.index, hourly_current[col], marker='s', label=col, linewidth=2)
axes[1].set_xlabel('Hodina dňa')
axes[1].set_ylabel('Priemerný prúd [A]')
axes[1].set_title('Denný profil prúdu (NN elektromery)')
axes[1].legend()
axes[1].set_xticks(range(24))
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_denny_profil.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Uložené: {OUTPUT_DIR}/04_denny_profil.png")

In [ ]:
# Sezónny profil - priemerné hodnoty cez jednotlivé mesiace odhalia ročnú sezónnosť
print("\n--- SEZÓNNY PROFIL (NN elektromery) ---")

if len(nn_data) > 0:
    monthly_voltage = nn_data.groupby('month')[voltage_cols].mean()
    monthly_current = nn_data.groupby('month')[current_cols].mean()
    
    print("\nPriemerné napätie podľa mesiaca:")
    print(monthly_voltage.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
months_sk = ['Jan', 'Feb', 'Mar', 'Apr', 'Máj', 'Jún', 'Júl', 'Aug', 'Sep', 'Okt', 'Nov', 'Dec']

for col in voltage_cols:
    axes[0].plot(monthly_voltage.index, monthly_voltage[col], marker='o', label=col, linewidth=2)
axes[0].set_xlabel('Mesiac')
axes[0].set_ylabel('Priemerné napätie [V]')
axes[0].set_title('Sezónny profil napätia (NN)')
axes[0].legend()
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(months_sk, rotation=45)
axes[0].grid(True, alpha=0.3)

for col in current_cols:
    axes[1].plot(monthly_current.index, monthly_current[col], marker='s', label=col, linewidth=2)
axes[1].set_xlabel('Mesiac')
axes[1].set_ylabel('Priemerný prúd [A]')
axes[1].set_title('Sezónny profil prúdu (NN)')
axes[1].legend()
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(months_sk, rotation=45)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_sezonny_profil.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Uložené: {OUTPUT_DIR}/05_sezonny_profil.png")

---
# ČASŤ 6: DISTRIBÚCIE HODNÔT
---

In [ ]:
print("="*70)
print("ČASŤ 6: DISTRIBÚCIE HODNÔT")
print("="*70)

# Zoberieme menšiu vzorku - na histogram s 100 binmi by milióny hodnôt boli zbytočné
sample_size = min(500_000, len(nn_data))
sample = nn_data.sample(sample_size, random_state=42) if len(nn_data) > 0 else df.sample(sample_size, random_state=42)

# Mriežka 2x3: hore napätia (3 fázy), dole prúdy (3 fázy)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Napätie - do každého histogramu pridáme čiaru s priemerom (červená) a mediánom (zelená)
# Ak sa priemer a medián výrazne líšia, distribúcia je šikmá
for i, col in enumerate(voltage_cols):
    axes[0, i].hist(sample[col], bins=100, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, i].axvline(sample[col].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {sample[col].mean():.1f}')
    axes[0, i].axvline(sample[col].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {sample[col].median():.1f}')
    axes[0, i].set_xlabel(f'{col} [V]')
    axes[0, i].set_ylabel('Frekvencia')
    axes[0, i].set_title(f'Distribúcia {col}')
    axes[0, i].legend(fontsize=9)

# Prúd - to isté, len druhá farba
for i, col in enumerate(current_cols):
    axes[1, i].hist(sample[col], bins=100, color='coral', edgecolor='black', alpha=0.7)
    axes[1, i].axvline(sample[col].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {sample[col].mean():.2f}')
    axes[1, i].axvline(sample[col].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {sample[col].median():.2f}')
    axes[1, i].set_xlabel(f'{col} [A]')
    axes[1, i].set_ylabel('Frekvencia')
    axes[1, i].set_title(f'Distribúcia {col}')
    axes[1, i].legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06_distribuce_hodnot.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Uložené: {OUTPUT_DIR}/06_distribuce_hodnot.png")

In [ ]:
# Boxploty - rýchle porovnanie, či sa všetky tri fázy správajú podobne
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# whis=3.0 = fúzy sú 3x IQR (rovnaké kritérium ako pri odstraňovaní outlierov v pipeline)
sample[voltage_cols].boxplot(ax=axes[0], whis = 3.0)
axes[0].set_ylabel('Napätie [V]')
axes[0].set_title('Porovnanie napätia medzi fázami (NN)')

sample[current_cols].boxplot(ax=axes[1], whis = 3.0)
axes[1].set_ylabel('Prúd [A]')
axes[1].set_title('Porovnanie prúdu medzi fázami (NN)')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/07_boxploty_fazy.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Uložené: {OUTPUT_DIR}/07_boxploty_fazy.png")

---
# ČASŤ 7: KORELAČNÁ ANALÝZA
---

In [ ]:
print("="*70)
print("ČASŤ 7: KORELAČNÁ ANALÝZA")
print("="*70)

# Korelačná matica - ukáže, ako spolu súvisia napätia a prúdy navzájom
corr_matrix = sample[voltage_cols + current_cols].corr()
print("\nKorelačná matica:")
print(corr_matrix.round(4).to_string())

# Heatmapa - červená = pozitívna korelácia, modrá = negatívna, biela = žiadna
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, 
            fmt='.3f', square=True, linewidths=0.5,
            cbar_kws={'label': 'Korelačný koeficient'})
plt.title('Korelačná matica - napätie a prúd (NN)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/08_korelacna_matica.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Uložené: {OUTPUT_DIR}/08_korelacna_matica.png")

---
# ČASŤ 8: ANALÝZA SEGMENTOV
---

In [ ]:
print("="*70)
print("ČASŤ 8: ANALÝZA SEGMENTOV")
print("="*70)

# Segment = súvislý úsek meraní bez väčšej medzery a bez zmeny stavu Chyba
# (jeden segment = jeden "vstup" pre model)
print(f"\nCelkovo segmentov: {len(segments):,}")
print(f"Unikátnych EIC: {segments['eic'].nunique():,}")

print("\nŠtatistiky dĺžky segmentov (dni):")
print(segments['duration_days'].describe().round(2).to_string())

if 'typ_siete' in segments.columns:
    print("\nRozdelenie podľa typu siete:")
    seg_by_type = segments.groupby('typ_siete', observed=True).agg(
        pocet_segmentov=('eic', 'count'),
        pocet_eic=('eic', 'nunique'),
        priemer_dlzka=('duration_days', 'mean'),
        median_dlzka=('duration_days', 'median'),
        min_dlzka=('duration_days', 'min'),
        max_dlzka=('duration_days', 'max'),
        trenovacich_prikladov=('num_training_examples', 'sum')
    ).round(2)
    print(seg_by_type.to_string())

In [ ]:
# Vizualizácia segmentov
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram dĺžky segmentov - väčšina býva krátkych, pár dlhých (pravostranne šikmé)
axes[0].hist(segments['duration_days'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(segments['duration_days'].mean(), color='red', linestyle='--', linewidth=2, 
                label=f"Mean: {segments['duration_days'].mean():.1f}d")
axes[0].axvline(segments['duration_days'].median(), color='green', linestyle='--', linewidth=2, 
                label=f"Median: {segments['duration_days'].median():.1f}d")
axes[0].set_xlabel('Dĺžka segmentu [dni]')
axes[0].set_ylabel('Frekvencia')
axes[0].set_title('Distribúcia dĺžky segmentov')
axes[0].legend()

# Koľko segmentov pripadá na jeden elektromer - niektoré majú 1, iné desiatky
segments_per_eic = segments.groupby('eic').size()
axes[1].hist(segments_per_eic.values, bins=range(1, segments_per_eic.max()+2), 
             color='coral', edgecolor='black', alpha=0.7, align='left')
axes[1].set_xlabel('Počet segmentov na EIC')
axes[1].set_ylabel('Počet EIC')
axes[1].set_title('Distribúcia počtu segmentov na elektromer')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/09_analyza_segmentov.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Uložené: {OUTPUT_DIR}/09_analyza_segmentov.png")

---
# ČASŤ 9: UKÁŽKA ČASOVÉHO RADU
---

In [ ]:
print("="*70)
print("ČASŤ 9: UKÁŽKA ČASOVÉHO RADU")
print("="*70)

# Vyberieme náhodný segment dlhý aspoň 14 dní (chceme vidieť aspoň dva týždenné cykly)
long_segments = segments[segments['duration_days'] >= 14]
if len(long_segments) > 0:
    sample_seg = long_segments.sample(1, random_state=42).iloc[0]
    sample_eic = sample_seg['eic']
    sample_seg_id = sample_seg['segment_id']
else:
    # Fallback ak nie je dosť dlhý segment - zoberieme najčastejší EIC
    sample_eic = df['eic'].mode()[0]
    sample_seg_id = 0

sample_meter = df[(df['eic'] == sample_eic) & (df['segment_id'] == sample_seg_id)].copy()
sample_meter = sample_meter.sort_values('t_utc')

print(f"\nVybraný elektromer: {sample_eic}")
print(f"Segment ID: {sample_seg_id}")
print(f"Počet záznamov: {len(sample_meter):,}")
print(f"Časový rozsah: {sample_meter['t_utc'].min()} - {sample_meter['t_utc'].max()}")

In [ ]:
# Vykreslíme len 7-dňový výsek (na celom segmente by sa krivky zlepili)
start_date = sample_meter['t_utc'].min()
end_date = start_date + pd.Timedelta(days=7)
week_data = sample_meter[(sample_meter['t_utc'] >= start_date) & (sample_meter['t_utc'] < end_date)]

fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# Napätie - tri fázy
for col in voltage_cols:
    axes[0].plot(week_data['t_utc'], week_data[col], label=col, alpha=0.8, linewidth=1)
axes[0].set_ylabel('Napätie [V]')
axes[0].set_title(f'Časový rad napätia - elektromer {sample_eic} (7 dní)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Prúd - tu by mali byť vidieť denné špičky
for col in current_cols:
    axes[1].plot(week_data['t_utc'], week_data[col], label=col, alpha=0.8, linewidth=1)
axes[1].set_xlabel('Čas')
axes[1].set_ylabel('Prúd [A]')
axes[1].set_title(f'Časový rad prúdu - elektromer {sample_eic} (7 dní)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/10_casovy_rad_ukazka.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Uložené: {OUTPUT_DIR}/10_casovy_rad_ukazka.png")

---
# ČASŤ 10: AUTOKORELAČNÁ ANALÝZA
---

In [ ]:
print("="*70)
print("ČASŤ 10: AUTOKORELAČNÁ ANALÝZA")
print("="*70)

# ACF a PACF - nástroje na hľadanie periodicity v časovom rade
# (potrebné napr. pre nastavenie SARIMA)
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Berieme 336 hodnôt = 14 dní pri 1h granularite
acf_data = sample_meter['u1_norm'].dropna().values[:336]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ACF napätia - 48 lagov = 2 dni, peak na lagu 24 by potvrdil dennú periodicitu
plot_acf(acf_data, lags=48, ax=axes[0, 0], alpha=0.05)
axes[0, 0].set_title('ACF - Napätie u1_norm (48 lagov = 2 dni)')
axes[0, 0].set_xlabel('Lag (hodinové intervaly)')

# PACF napätia
plot_pacf(acf_data, lags=30, ax=axes[0, 1], method='ywm')
axes[0, 1].set_title('PACF - Napätie u1_norm')
axes[0, 1].set_xlabel('Lag')

# To isté pre prúd
acf_current = sample_meter['i1_norm'].dropna().values[:336]
plot_acf(acf_current, lags=48, ax=axes[1, 0], alpha=0.05)
axes[1, 0].set_title('ACF - Prúd i1_norm (48 lagov = 2 dni)')
axes[1, 0].set_xlabel('Lag (hodinové intervaly)')

plot_pacf(acf_current, lags=30, ax=axes[1, 1], method='ywm')
axes[1, 1].set_title('PACF - Prúd i1_norm')
axes[1, 1].set_xlabel('Lag')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/11_autokorelacia.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Uložené: {OUTPUT_DIR}/11_autokorelacia.png")

print("\nInterpretácia (pre 1h granularitu):")
print("  - Lag 1 = 1 hodina")
print("  - Lag 24 = 1 deň (denná periodicita)")
print("  - Lag 168 = 7 dní (týždenná periodicita)")

---
# ČASŤ 11: ANALÝZA PORUCHOVÝCH DÁT
---

In [ ]:
print("="*70)
print("ČASŤ 11: ANALÝZA PORUCHOVÝCH DÁT")
print("="*70)

print(f"\nPoruchové záznamy (Chyba=1): {len(fault_data):,}")
print(f"EIC s poruchami: {fault_data['eic'].nunique()}")

# Pre každú kombináciu (eic, segment_id) zistíme, kedy porucha začala, skončila a ako dlho trvala
fault_segments = fault_data.groupby(['eic', 'segment_id'], observed=True).agg(
    start_time=('t_utc', 'min'),
    end_time=('t_utc', 'max'),
    num_records=('t_utc', 'count')
).reset_index()
fault_segments['duration_hours'] = (fault_segments['end_time'] - fault_segments['start_time']).dt.total_seconds() / 3600
fault_segments['duration_days'] = fault_segments['duration_hours'] / 24

print(f"\nPoruchové segmenty: {len(fault_segments):,}")
print(f"\nŠtatistiky dĺžky poruchových segmentov (dni):")
print(fault_segments['duration_days'].describe().round(2).to_string())


In [ ]:
# Porovnanie štatistík - kľúčová otázka: líšia sa hodnoty počas poruchy od normálu?
print("\n--- POROVNANIE: NORMÁLNE vs PORUCHOVÉ DÁTA ---")
print("\nNormálne dáta (Chyba=0):")
print(nn_data[voltage_cols + current_cols].describe().round(4).to_string())

print("\nPoruchové dáta (Chyba=1):")
if len(fault_data) > 0:
    print(fault_data[voltage_cols + current_cols].describe().round(4).to_string())
else:
    print("Žiadne poruchové dáta.")


In [ ]:
# Vizuálne porovnanie distribúcií - prekrývané histogramy normálnych a poruchových hodnôt
# Ak sa modré a červené krivky výrazne prekrývajú, znamená to, že poruchy sa
# v hodnotách nelíšia od normálu - vtedy bude anomaly detection ťažká
if len(fault_data) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('Porovnanie distribúcií: Normálne (Chyba=0) vs Poruchové (Chyba=1)', fontsize=14, fontweight='bold')

    # density=True normalizuje histogram, aby sa dalo porovnávať aj pri rôznom počte vzoriek
    sample_normal = nn_data.sample(min(200_000, len(nn_data)), random_state=42)
    sample_fault = fault_data.sample(min(200_000, len(fault_data)), random_state=42)

    for i, col in enumerate(voltage_cols):
        axes[0, i].hist(sample_normal[col], bins=80, alpha=0.6, color='steelblue', label='Normálne', density=True)
        axes[0, i].hist(sample_fault[col], bins=80, alpha=0.6, color='red', label='Poruchy', density=True)
        axes[0, i].set_title(f'{col}')
        axes[0, i].set_xlabel('Napätie [V]')
        axes[0, i].legend(fontsize=9)

    for i, col in enumerate(current_cols):
        axes[1, i].hist(sample_normal[col], bins=80, alpha=0.6, color='steelblue', label='Normálne', density=True)
        axes[1, i].hist(sample_fault[col], bins=80, alpha=0.6, color='red', label='Poruchy', density=True)
        axes[1, i].set_title(f'{col}')
        axes[1, i].set_xlabel('Prúd [A]')
        axes[1, i].legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/12_normal_vs_poruchy_distribuce.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Uložené: {OUTPUT_DIR}/12_normal_vs_poruchy_distribuce.png")


In [ ]:
# Boxploty vedľa seba: normálne (modré) a poruchové (červené) pre každú fázu
if len(fault_data) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Napätie
    bp_data_v = [nn_data[col].dropna().sample(min(100_000, len(nn_data)), random_state=42) for col in voltage_cols]
    bp_fault_v = [fault_data[col].dropna().sample(min(100_000, len(fault_data)), random_state=42) for col in voltage_cols]
    
    # Trik na pozície: modré krabice na 1,4,7 a červené na 2,5,8 (vedľa seba v dvojiciach)
    positions_n = [1, 4, 7]
    positions_f = [2, 5, 8]
    
    bp1 = axes[0].boxplot(bp_data_v, positions=positions_n, widths=0.8, whis=3.0,
                          patch_artist=True, boxprops=dict(facecolor='steelblue', alpha=0.6))
    bp2 = axes[0].boxplot(bp_fault_v, positions=positions_f, widths=0.8, whis=3.0,
                          patch_artist=True, boxprops=dict(facecolor='red', alpha=0.6))
    # Labely umiestnime doprostred dvojice (medzi 1 a 2 je 1.5)
    axes[0].set_xticks([1.5, 4.5, 7.5])
    axes[0].set_xticklabels(voltage_cols)
    axes[0].set_ylabel('Napätie [V]')
    axes[0].set_title('Napätie: Normálne vs Poruchy')
    axes[0].legend([bp1["boxes"][0], bp2["boxes"][0]], ['Normálne', 'Poruchy'], fontsize=9)

    # Prúd - to isté
    bp_data_c = [nn_data[col].dropna().sample(min(100_000, len(nn_data)), random_state=42) for col in current_cols]
    bp_fault_c = [fault_data[col].dropna().sample(min(100_000, len(fault_data)), random_state=42) for col in current_cols]
    
    bp3 = axes[1].boxplot(bp_data_c, positions=positions_n, widths=0.8, whis=3.0,
                          patch_artist=True, boxprops=dict(facecolor='steelblue', alpha=0.6))
    bp4 = axes[1].boxplot(bp_fault_c, positions=positions_f, widths=0.8, whis=3.0,
                          patch_artist=True, boxprops=dict(facecolor='red', alpha=0.6))
    axes[1].set_xticks([1.5, 4.5, 7.5])
    axes[1].set_xticklabels(current_cols)
    axes[1].set_ylabel('Prúd [A]')
    axes[1].set_title('Prúd: Normálne vs Poruchy')
    axes[1].legend([bp3["boxes"][0], bp4["boxes"][0]], ['Normálne', 'Poruchy'], fontsize=9)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/13_normal_vs_poruchy_boxplot.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Uložené: {OUTPUT_DIR}/13_normal_vs_poruchy_boxplot.png")


In [ ]:
# Časový priebeh konkrétneho EIC s poruchou - ukáže prechod z normálu do poruchy v reálnom čase
if len(fault_data) > 0:
    # Hľadáme EIC s vyváženým pomerom normálne/poruchové (30-70%) - tam bude jasný prechod
    candidate_stats = (df.groupby('eic')
                         .agg(n_total=('Chyba', 'size'),
                              n_fault=('Chyba', 'sum'))
                         .reset_index())
    candidate_stats['n_normal'] = candidate_stats['n_total'] - candidate_stats['n_fault']
    candidate_stats['fault_ratio'] = candidate_stats['n_fault'] / candidate_stats['n_total']
    
    candidates = candidate_stats[
        (candidate_stats['fault_ratio'] >= 0.3) &
        (candidate_stats['fault_ratio'] <= 0.7) &
        (candidate_stats['n_total'] >= 1000)
    ].sort_values('n_total', ascending=False)
    
    if len(candidates) > 0:
        fault_eic = candidates.iloc[2]['eic']
        print(f"Vybraný EIC (vyvážený normálne/poruchové): {fault_eic}")
        print(f"  Poruchový pomer: {candidates.iloc[0]['fault_ratio']:.1%}")
    else:
        # Fallback - ak nikto neprejde filtrami, zoberieme aspoň najdlhší fault segment
        fault_eic = fault_segments.sort_values('duration_days', ascending=False).iloc[0]['eic']
        print(f"Fallback - najdlhší fault segment: {fault_eic}")
    
    # Celý časový rad zvoleného EIC
    eic_data = df[df['eic'] == fault_eic].sort_values('t_utc').copy().reset_index(drop=True)
    
    print(f"\nUkážka EIC s poruchou: {fault_eic}")
    print(f"  Celkovo záznamov: {len(eic_data):,}")
    print(f"  Normálnych: {(eic_data['Chyba']==0).sum():,}")
    print(f"  Poruchových: {(eic_data['Chyba']==1).sum():,}")
    
    # Hľadáme bod, kde Chyba prešla z 0 na 1 alebo opačne
    # diff() vráti rozdiel medzi po sebe idúcimi hodnotami - tam kde nie je 0, je prechod
    chyba_diff = eic_data['Chyba'].diff().fillna(0)
    transition_idx = eic_data.index[chyba_diff != 0].tolist()
    
    if len(transition_idx) > 0:
        # Vyberieme stredný prechod (prvý/posledný môže byť na okraji datasetu)
        best_idx = transition_idx[len(transition_idx) // 2]
        transition_time = eic_data.loc[best_idx, 't_utc']
        
        # Okno +-15 dní okolo prechodu
        window_days = 15
        t_start = transition_time - pd.Timedelta(days=window_days)
        t_end = transition_time + pd.Timedelta(days=window_days)
        
        plot_data = eic_data[(eic_data['t_utc'] >= t_start) & 
                             (eic_data['t_utc'] <= t_end)].copy()
        
        print(f"  Prechod: {transition_time}")
        print(f"  Okno: {t_start.date()} az {t_end.date()}")
        print(f"  Zaznamov v okne: {len(plot_data):,}")
    else:
        # Ak nie je žiadny prechod, zoberieme strednú polovicu radu
        plot_data = eic_data.iloc[len(eic_data)//4 : 3*len(eic_data)//4].copy()
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 8))
    
    # Rozdelíme dáta na normálne a poruchové, vykreslíme inou farbou
    normal = plot_data[plot_data['Chyba'] == 0]
    fault = plot_data[plot_data['Chyba'] == 1]
    
    # Napätie
    axes[0].plot(normal['t_utc'], normal['u1_norm'], color='steelblue', alpha=0.8, linewidth=0.7, label='Normálne')
    axes[0].plot(fault['t_utc'], fault['u1_norm'], color='red', alpha=0.7, linewidth=0.7, label='Porucha')
    axes[0].set_ylabel('Napätie U1 [V]', fontsize=15)
    axes[0].set_title(f'Časový rad s poruchou - EIC {fault_eic}', fontsize=16)
    axes[0].legend(loc='best', fontsize=15)
    axes[0].tick_params(axis='both', labelsize=15)
    axes[0].grid(True, alpha=0.3)
    
    # Prúd
    axes[1].plot(normal['t_utc'], normal['i1_norm'], color='steelblue', alpha=0.8, linewidth=0.7, label='Normálne')
    axes[1].plot(fault['t_utc'], fault['i1_norm'], color='red', alpha=0.7, linewidth=0.7, label='Porucha')
    axes[1].set_ylabel('Prúd I1 [A]', fontsize=15)
    axes[1].set_xlabel('Čas', fontsize=20)
    axes[1].legend(loc='best', fontsize=15)
    axes[1].tick_params(axis='both', labelsize=15)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/14_casovy_rad_porucha.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Uložené: {OUTPUT_DIR}/14_casovy_rad_porucha.png")

---
# ZÁVEREČNÝ SÚHRN
---

In [ ]:
print("="*70)
print("ZÁVEREČNÝ SÚHRN PRE DIPLOMOVÚ PRÁCU")
print("="*70)

# Poistka pre prípad, že nejaký subset je prázdny
nn_count = len(nn_data) if len(nn_data) > 0 else 0
fault_count = len(fault_data) if len(fault_data) > 0 else 0
nn_eic = nn_data['eic'].nunique() if len(nn_data) > 0 else 0
fault_eic_count = fault_data['eic'].nunique() if len(fault_data) > 0 else 0

summary = f"""
================================================================================
                    SÚHRN DATASETU PRE DIPLOMOVÚ PRÁCU
================================================================================

1. ZDROJOVÉ DÁTA:
────────────────────────────────────────────────────────────────────────────────
   Dataset: tuke_poruchy_vzorky
     • Merania (parquet) + Evidencia porúch (CSV)
     • Poruchy priradené podľa dátumových rozsahov

2. SPRACOVANÝ DATASET (1h granularita):
────────────────────────────────────────────────────────────────────────────────
   • Celkový počet záznamov: {len(df):,}
   • Počet unikátnych EIC: {df['eic'].nunique():,}
   • Časový rozsah: {df['t_utc'].min()} až {df['t_utc'].max()}
   • Celková dĺžka: {(df['t_utc'].max() - df['t_utc'].min()).days:,} dní

3. ROZDELENIE PODĽA PRÍZNAKU CHYBA:
────────────────────────────────────────────────────────────────────────────────
   Normálne dáta (Chyba=0): {nn_count:,} záznamov, {nn_eic:,} EIC
   Poruchové dáta (Chyba=1): {fault_count:,} záznamov, {fault_eic_count} EIC

4. SEGMENTY (úseky bez porúch ≥ 8 dní):
────────────────────────────────────────────────────────────────────────────────
   • Počet segmentov: {len(segments):,}
   • Počet EIC: {segments["eic"].nunique():,}
   • Priemerná dĺžka: {segments["duration_days"].mean():.1f} dní
   • Medián dĺžky: {segments["duration_days"].median():.1f} dní
   • Celkovo trénovacích príkladov: {segments["num_training_examples"].sum():,}

5. EXPORTOVANÉ GRANULARITY:
────────────────────────────────────────────────────────────────────────────────
   • 10 minút (originál)
   • 30 minút
   • 1 hodina

================================================================================
"""

print(summary)

# Uložíme aj do textového súboru, aby sa dal použiť mimo notebooku
with open(f'{OUTPUT_DIR}/sumar_pre_DP.txt', 'w', encoding='utf-8') as f:
    f.write(summary)
print(f"\nSúhrn uložený do: {OUTPUT_DIR}/sumar_pre_DP.txt")


In [ ]:
import glob

print("\n" + "="*70)
print("VYGENEROVANÉ SÚBORY:")
print("="*70)

# Vypíšeme všetky vygenerované súbory aj s ich veľkosťou (KB)
files = sorted(glob.glob(f'{OUTPUT_DIR}/*'))
for f in files:
    size = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):40s} ({size:.1f} KB)")